In [1]:
!pip install -q google-genai supabase pillow

In [10]:


from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini connected successfully!")



Gemini connected successfully!


In [47]:


from google.colab import files
from PIL import Image
import io

# Upload an image from your computer
uploaded = files.upload()

# Read the uploaded image
filename = next(iter(uploaded))
image = Image.open(io.BytesIO(uploaded[filename]))

# Display the image
display(image)


Output hidden; open in https://colab.research.google.com to view.

In [48]:

from google.genai import errors
import time

models = [
    "gemini-3.1-flash-lite",
    "gemini-3-flash-preview",
    "gemini-3.8-flash"
]

prompt = """
Extract all visible text from this image.
Preserve spelling, names, dates and numbers.
Do not invent missing information.
Mark unreadable text as [UNREADABLE].
"""

extracted_text = None

for model_name in models:
    for attempt in range(2):
        try:
            print(f"Trying {model_name}...")

            response = client.models.generate_content(
                model=model_name,
                contents=[prompt, image]
            )

            if not response.text:
                raise ValueError("Empty model response")

            extracted_text = response.text
            print(f"Success with {model_name}")
            break

        except errors.ServerError as e:
            print(f"Temporary server error: {e}")
            if attempt == 0:
                time.sleep(5)

        except errors.ClientError as e:
            print(f"API error for {model_name}: {e}")
            break

    if extracted_text is not None:
        break

if extracted_text is not None:
    print("\nEXTRACTED TEXT:")
    print(extracted_text)
else:
    print("Extraction failed. Try again later.")


Trying gemini-3.1-flash-lite...
Success with gemini-3.1-flash-lite

EXTRACTED TEXT:
winners
VAT INVOICE
WINNERS ST PAUL PHOENIX
Route de St Paul Phoenix
PICK & BUY LTD
VAT:VAT20032249 / BRN:C06003539
Tel:490-2126 Fax:696-9410
LOYALTY ID: 16833323

1> SAC BIO CAISSE 4.00
1> SAC BIO CAISSE 4.00
2> S.SHOP CARCASSE PLT 88.92
2> S.SHOP CARCASSE PLT 93.25
2> MARILYN C MINCE 400G 87.95
1> SUBANA V COOKI 225G 58.00
1> HUP SENG CREAM CRACK 109.00
1> RYLTY DIGEST BI 400G 99.95 P
3> WEETABIX WHL WHEAT C 155.47
2> CHANTECLER FOIE POUL 95.90 P
2> CHANTECLER FOIE POUL 95.90 P
2> MANDARINE SWEET C FI 119.95 P

TOTAL 1012.29 Rs
12 Article(s)

SBN 1012.29 R
Numero Ticket 021361

IVA incluse:
2 VATZRO 581.87 0.00
1 VAT15% 274.95 35.87
3 VATEXP 155.47 0.00

Ope Mag. Date Heure TPV Ticket
112 18 11.11.26 16:59 9 342831

Check your rewards at winners.mu

On est tous WINNERS

Vous pouvez desormais faire vos courses
en ligne sur www.winners.mu
Les clients doivent désormais présenter
le ticket original pour l

In [22]:

%pip install -q supabase


In [27]:

import json
from google.genai import types

classification_prompt = f"""
You are an intelligent document classification
and information extraction system.

Analyze the following extracted text:

{extracted_text}

TASKS:
1. Identify the document type.
2. Extract all meaningful fields.
3. Standardize dates to YYYY-MM-DD when unambiguous.
4. Preserve original names, identifiers and numbers.
5. Identify any uncertain or unreadable information.

Allowed document types:
- receipt
- invoice
- form
- medical_report
- identity_document
- letter
- table
- handwritten_note
- other

Return JSON with this structure:

{{
  "document_type": "invoice",
  "fields": [
    {{"name": "invoice_number", "value": "INV-001"}},
    {{"name": "date", "value": "2026-11-13"}},
    {{"name": "total_amount", "value": "1500"}}
  ],
  "uncertain_parts": []
}}

Use field names in snake_case.
Only include information present in the text.
Do not invent missing values.
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=classification_prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json"
    )
)

structured_data = json.loads(response.text)

print(json.dumps(
    structured_data,
    indent=2,
    ensure_ascii=False
))


{
  "document_type": "invoice",
  "fields": [
    {
      "name": "business_registration_number",
      "value": "F07000041"
    },
    {
      "name": "vat_registration_number",
      "value": "VAT22000591"
    },
    {
      "name": "issuer_name",
      "value": "Central Electricity Board"
    },
    {
      "name": "invoice_number",
      "value": "250012106216"
    },
    {
      "name": "invoice_date",
      "value": "2026-10-19"
    },
    {
      "name": "customer_name",
      "value": "OUMESH THUMMANAH"
    },
    {
      "name": "business_partner_id",
      "value": "8144"
    },
    {
      "name": "contract_account",
      "value": "100075358"
    },
    {
      "name": "tariff",
      "value": "140"
    },
    {
      "name": "supply_address",
      "value": "NO 10 RUE MELLISH, FOREST SIDE"
    },
    {
      "name": "meter_number",
      "value": "000099080193"
    },
    {
      "name": "current_reading",
      "value": "35,654"
    },
    {
      "name": "previous_readin

In [45]:

response = requests.get(
    url + "/rest/v1/extracted_documents",
    headers={
        "apikey": key,
        "Authorization": f"Bearer {key}"
    },
    params={
        "select": "id,source_filename,document_type,review_status",
        "order": "created_at.desc",
        "limit": "10"
    },
    timeout=15
)

response.raise_for_status()

for row in response.json():
    print(row)


{'id': 1, 'source_filename': 'test1.jpeg', 'document_type': 'invoice', 'review_status': 'pending_review'}


In [49]:

import json
from google.genai import types

grocery_prompt = f"""
You are a grocery receipt extraction system for Mauritius.

Extract structured grocery information from this OCR text:

{extracted_text}

Return a JSON object with:
- store_name
- receipt_date (YYYY-MM-DD or null)
- currency ("MUR")
- receipt_total
- items (array)
- uncertain_parts (array)

For each item include:
- original_name: exact product text
- normalized_name: understandable product name
- quantity: number of purchased units
- unit: kg, g, ml, l, piece, or unknown
- package_size: number or null
- package_unit: unit of package size or null
- line_total: price in Mauritian rupees or null
- confidence: high, medium, or low

Rules:
1. Never invent products or prices.
2. Preserve duplicate receipt lines.
3. Do not confuse product quantity with package size.
4. Do not assume a price is a unit price.
5. Mark ambiguous products as low confidence.
6. Do not include VAT summaries as products.
7. Use null when a value cannot be determined.
8. Return only valid JSON.
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=grocery_prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json"
    )
)

grocery_data = json.loads(response.text)

print(json.dumps(
    grocery_data,
    indent=2,
    ensure_ascii=False
))


{
  "store_name": "WINNERS ST PAUL PHOENIX",
  "receipt_date": "2026-11-11",
  "currency": "MUR",
  "receipt_total": 1012.29,
  "items": [
    {
      "original_name": "SAC BIO CAISSE",
      "normalized_name": "Shopping Bag",
      "quantity": 1,
      "unit": "piece",
      "package_size": null,
      "package_unit": null,
      "line_total": 4.0,
      "confidence": "high"
    },
    {
      "original_name": "SAC BIO CAISSE",
      "normalized_name": "Shopping Bag",
      "quantity": 1,
      "unit": "piece",
      "package_size": null,
      "package_unit": null,
      "line_total": 4.0,
      "confidence": "high"
    },
    {
      "original_name": "S.SHOP CARCASSE PLT",
      "normalized_name": "Chicken Carcass",
      "quantity": 2,
      "unit": "piece",
      "package_size": null,
      "package_unit": null,
      "line_total": 88.92,
      "confidence": "medium"
    },
    {
      "original_name": "S.SHOP CARCASSE PLT",
      "normalized_name": "Chicken Carcass",
      "quant

In [50]:

import requests

response = requests.get(
    url + "/rest/v1/pantry_items",
    headers={
        "apikey": key,
        "Authorization": f"Bearer {key}"
    },
    params={"select": "*", "limit": 5},
    timeout=15
)

print("HTTP status:", response.status_code)
print("Response:", response.text)


HTTP status: 200
Response: []


In [ ]:

from supabase import create_client
from getpass import getpass

# Reuse your existing Supabase URL and anon key.
# Replace these names if your notebook uses different variables.
SUPABASE_URL = url
SUPABASE_ANON_KEY = key

email = input("Demo user email: ")
password = getpass("Demo user password: ")

supabase = create_client(
    SUPABASE_URL,
    SUPABASE_ANON_KEY
)

auth_response = supabase.auth.sign_in_with_password({
    "email": email,
    "password": password
})

user_id = auth_response.user.id

print("Authentication successful")
print("User ID:", user_id)


In [55]:

from urllib.parse import urlparse

print("Connected Supabase project:")
print(urlparse(url).hostname)


Connected Supabase project:
rfcbwhesohomqcplzqwj.supabase.co


In [56]:

import requests
from getpass import getpass

email = input("Demo email: ").strip()
password = getpass("Demo password: ")

response = requests.post(
    url.rstrip("/") + "/auth/v1/token",
    params={"grant_type": "password"},
    headers={
        "apikey": key,
        "Content-Type": "application/json"
    },
    json={
        "email": email,
        "password": password
    },
    timeout=20
)

print("HTTP status:", response.status_code)

if response.ok:
    print("Authentication successful!")
else:
    print("Error:", response.text)


Demo email: manvi@gmail.com
Demo password: ··········
HTTP status: 200
Authentication successful!


In [57]:

from supabase import create_client

supabase = create_client(url, key)

auth = supabase.auth.sign_in_with_password({
    "email": email,
    "password": password
})

user_id = auth.user.id

print("Connected successfully!")
print("User ID:", user_id)


Connected successfully!
User ID: 71d8eafb-0035-412b-90ac-ad8a7f44f14d


In [58]:

household_data = {
    "owner_id": user_id,
    "name": "LiveBetter Demo Household",
    "members": 4,
    "currency": "MUR",
    "preferences": {
        "diet": "none",
        "allergies": [],
        "servings": 4
    },
    "budget_goals": {
        "groceries": 12000,
        "electricity": 2500,
        "water": 500
    }
}

existing = (
    supabase.table("households")
    .select("*")
    .eq("owner_id", user_id)
    .eq("name", "LiveBetter Demo Household")
    .execute()
)

if existing.data:
    household = existing.data[0]
    print("Existing household found")
else:
    result = (
        supabase.table("households")
        .insert(household_data)
        .execute()
    )
    household = result.data[0]
    print("New household created")

household_id = household["id"]

print("Household ID:", household_id)
print("Household name:", household["name"])


New household created
Household ID: fe951ebe-54ab-42ec-9868-07ee674a90fa
Household name: LiveBetter Demo Household


In [59]:

result = (
    supabase.table("households")
    .select("id,name,members,currency")
    .eq("id", household_id)
    .execute()
)

print(result.data)


[{'id': 'fe951ebe-54ab-42ec-9868-07ee674a90fa', 'name': 'LiveBetter Demo Household', 'members': 4, 'currency': 'MUR'}]


In [61]:

import uuid

test_receipt = {
    "p_household_id": household_id,
    "p_source_identifier": str(uuid.uuid4()),
    "p_retailer": "LiveBetter Demo Store",
    "p_receipt_date": "2026-11-13",
    "p_total": 233,
    "p_items": [
        {
            "original_name": "RICE 2KG",
            "normalized_name": "rice",
            "quantity": 2,
            "unit": "kg",
            "line_total": 120
        },
        {
            "original_name": "CARROTS 1KG",
            "normalized_name": "carrots",
            "quantity": 1,
            "unit": "kg",
            "line_total": 65
        },
        {
            "original_name": "EGGS 6",
            "normalized_name": "eggs",
            "quantity": 6,
            "unit": "piece",
            "line_total": 48
        }
    ]
}

result = supabase.rpc(
    "confirm_grocery_receipt",
    test_receipt
).execute()

print("Receipt saved successfully!")
print("Receipt ID:", result.data)


Receipt saved successfully!
Receipt ID: 8c5ec47f-0583-496d-a22d-ac2bd729734d


In [62]:

from postgrest.exceptions import APIError

try:
    result = supabase.rpc(
        "confirm_grocery_receipt",
        test_receipt
    ).execute()

    print("Receipt confirmation successful")
    print("Receipt ID:", result.data)

except APIError as e:
    print("Database error:", e)
    raise


Receipt confirmation successful
Receipt ID: 8c5ec47f-0583-496d-a22d-ac2bd729734d


In [63]:

from decimal import Decimal, InvalidOperation
import json

ALLOWED_UNITS = {"kg", "g", "l", "ml", "piece"}

def normalize_receipt(grocery_data):
    items = []

    for product in grocery_data.get("items", []):
        name = str(
            product.get("normalized_name")
            or product.get("original_name")
            or ""
        ).strip().lower()

        unit = str(
            product.get("unit") or "unknown"
        ).strip().lower()

        try:
            quantity = Decimal(
                str(product.get("quantity"))
            )
        except (InvalidOperation, TypeError):
            quantity = Decimal("0")

        raw_price = product.get("line_total")
        try:
            price = (
                None if raw_price is None
                else float(Decimal(str(raw_price)))
            )
        except (InvalidOperation, TypeError):
            price = None

        if not name or quantity <= 0:
            print("Needs review:", product)
            continue

        if unit not in ALLOWED_UNITS:
            print("Unsupported unit:", product)
            continue

        if price is not None and price < 0:
            print("Invalid price:", product)
            continue

        items.append({
            "original_name": str(
                product.get("original_name") or name
            ),
            "normalized_name": name,
            "quantity": float(quantity),
            "unit": unit,
            "line_total": price
        })

    return items

normalized_items = normalize_receipt(grocery_data)

print(json.dumps(
    normalized_items,
    indent=2,
    ensure_ascii=False
))


[
  {
    "original_name": "SAC BIO CAISSE",
    "normalized_name": "shopping bag",
    "quantity": 1.0,
    "unit": "piece",
    "line_total": 4.0
  },
  {
    "original_name": "SAC BIO CAISSE",
    "normalized_name": "shopping bag",
    "quantity": 1.0,
    "unit": "piece",
    "line_total": 4.0
  },
  {
    "original_name": "S.SHOP CARCASSE PLT",
    "normalized_name": "chicken carcass",
    "quantity": 2.0,
    "unit": "piece",
    "line_total": 88.92
  },
  {
    "original_name": "S.SHOP CARCASSE PLT",
    "normalized_name": "chicken carcass",
    "quantity": 2.0,
    "unit": "piece",
    "line_total": 93.25
  },
  {
    "original_name": "MARILYN C MINCE 400G",
    "normalized_name": "marilyn chicken mince",
    "quantity": 2.0,
    "unit": "piece",
    "line_total": 87.95
  },
  {
    "original_name": "SUBANA V COOKI 225G",
    "normalized_name": "subana vegetable cookies",
    "quantity": 1.0,
    "unit": "piece",
    "line_total": 58.0
  },
  {
    "original_name": "HUP SENG CR

In [64]:

import pandas as pd
from IPython.display import display

df = pd.DataFrame(normalized_items)

display(df)


,original_name,normalized_name,quantity,unit,line_total
0,SAC BIO CAISSE,shopping bag,1.0,piece,4.00
1,SAC BIO CAISSE,shopping bag,1.0,piece,4.00
2,S.SHOP CARCASSE PLT,chicken carcass,2.0,piece,88.92
3,S.SHOP CARCASSE PLT,chicken carcass,2.0,piece,93.25
4,MARILYN C MINCE 400G,marilyn chicken mince,2.0,piece,87.95
5,SUBANA V COOKI 225G,subana vegetable cookies,1.0,piece,58.00
6,HUP SENG CREAM CRACK,hup seng cream crackers,1.0,piece,109.00
7,RYLTY DIGEST BI 400G,royalty digestive biscuits,1.0,piece,99.95
8,WEETABIX WHL WHEAT C,weetabix whole wheat cereal,3.0,piece,155.47
9,CHANTECLER FOIE POUL,chantecler chicken liver,2.0,piece,95.90


In [65]:

import hashlib
import json

if not normalized_items:
    raise ValueError("No valid grocery items to confirm.")

# Use the original receipt image hash when available.
# For this Colab test, generate a stable identifier
# from the extracted document content.
source_text = extracted_text

source_identifier = hashlib.sha256(
    source_text.encode("utf-8")
).hexdigest()

receipt_payload = {
    "p_household_id": household_id,
    "p_source_identifier": source_identifier,
    "p_retailer": grocery_data.get("store_name"),
    "p_receipt_date": grocery_data.get("receipt_date"),
    "p_total": grocery_data.get("receipt_total"),
    "p_items": normalized_items
}

print("Receipt ready for confirmation")
print(json.dumps(
    receipt_payload,
    indent=2,
    default=str
))

confirmation = input(
    "Type CONFIRM to save the receipt: "
)

if confirmation == "CONFIRM":
    result = supabase.rpc(
        "confirm_grocery_receipt",
        receipt_payload
    ).execute()

    print("Receipt saved:", result.data)
else:
    print("Receipt not saved.")


Receipt ready for confirmation
{
  "p_household_id": "fe951ebe-54ab-42ec-9868-07ee674a90fa",
  "p_source_identifier": "e95e5f0d48f1e9509fcbb38a808a841ee398482a0801b87819b9343f3c89b1e1",
  "p_retailer": "WINNERS ST PAUL PHOENIX",
  "p_receipt_date": "2026-11-11",
  "p_total": 1012.29,
  "p_items": [
    {
      "original_name": "SAC BIO CAISSE",
      "normalized_name": "shopping bag",
      "quantity": 1.0,
      "unit": "piece",
      "line_total": 4.0
    },
    {
      "original_name": "SAC BIO CAISSE",
      "normalized_name": "shopping bag",
      "quantity": 1.0,
      "unit": "piece",
      "line_total": 4.0
    },
    {
      "original_name": "S.SHOP CARCASSE PLT",
      "normalized_name": "chicken carcass",
      "quantity": 2.0,
      "unit": "piece",
      "line_total": 88.92
    },
    {
      "original_name": "S.SHOP CARCASSE PLT",
      "normalized_name": "chicken carcass",
      "quantity": 2.0,
      "unit": "piece",
      "line_total": 93.25
    },
    {
      "origina

In [66]:

confirmation = input(
    "Type CONFIRM to save the receipt: "
).strip().upper()

if confirmation == "CONFIRM":
    result = supabase.rpc(
        "confirm_grocery_receipt",
        receipt_payload
    ).execute()

    print("Receipt saved successfully!")
    print("Receipt ID:", result.data)
else:
    print("Receipt not saved.")


Type CONFIRM to save the receipt: confirm
Receipt saved successfully!
Receipt ID: 50db9799-a271-4792-934f-f5c9cfa2b45a


In [67]:

import pandas as pd
from IPython.display import display

pantry = (
    supabase.table("pantry_items")
    .select("ingredient_name,quantity,unit,available")
    .eq("household_id", household_id)
    .order("ingredient_name")
    .execute()
)

display(pd.DataFrame(pantry.data))


,ingredient_name,quantity,unit,available
0,carrots,1.0,kg,True
1,chantecler chicken liver,4.0,piece,True
2,chicken carcass,4.0,piece,True
3,eggs,6.0,piece,True
4,hup seng cream crackers,1.0,piece,True
5,mandarine sweet fruit,2.0,piece,True
6,marilyn chicken mince,2.0,piece,True
7,rice,2.0,kg,True
8,royalty digestive biscuits,1.0,piece,True
9,shopping bag,2.0,piece,True


In [ ]:

receipts = (
    supabase.table("grocery_receipts")
    .select("id,retailer,receipt_date,total")
    .eq("household_id", household_id)
    .execute()
)

display(pd.DataFrame(receipts.data))


In [102]:

for model in client.models.list():
    if "flash" in model.name.lower():
        print(model.name)


models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [105]:

MODEL_NAME = "gemini-2.5-flash"

print("Selected model:", MODEL_NAME)


Selected model: gemini-2.5-flash
